# Contact coordinates: design (.oas) and SEM (.jpg)

Reads one SEM tile and its matching `.oas` design file, and prints the contact-hole centre
coordinates from both.

- **Design** contacts: centroid of each polygon, in **nm**, in the design frame (y up).
- **SEM** contacts: found with a simple **Otsu threshold**. The centre is the centroid of each
  dark region, in **pixels** `(u, v)` = (column, row), with v pointing **down** and pixel
  centres at whole numbers. If you set `PIXEL_SIZE_NM`, the centres are also given in
  **tile-local nm** (origin at the image centre, y up).

The SEM and design coordinates are **not** aligned to each other yet. That is the job of
the later pipeline stages.

**How to use:** set the paths in the next cell and run all cells. Leave the paths as `None`
to run a demo on synthetic files.

In [1]:
# ---- Settings: edit these ---------------------------------------------------
OAS_PATH = None        # e.g. r"D:\data\tile_0001.oas"   (None -> synthetic demo)
JPG_PATH = None        # e.g. r"D:\data\tile_0001.jpg"   (None -> synthetic demo)
LAYER = (1, 0)         # (layer, datatype) of the contacts in the .oas file
CELL = None            # top cell name, or None if the file has a single top cell
DATABAR_ROWS = 0       # rows of SEM info bar at the bottom of the image to ignore
DARK_CONTACTS = True   # True if contacts look dark in the SEM image
MIN_AREA_PX = 20       # ignore dark specks smaller than this many pixels
PIXEL_SIZE_NM = None   # nm per pixel; set it to also get tile-local nm coordinates

In [2]:
import sys
import tempfile
from pathlib import Path

import pandas as pd

from affine_ransac.features.contact import detect_contacts
from affine_ransac.geometry.frames import pixel_to_tile_nm
from affine_ransac.io.design import list_layers, load_layout, read_contacts
from affine_ransac.io.sem_image import crop_databar, load_sem_image

pd.set_option("display.max_rows", 20)
pd.set_option("display.float_format", "{:.2f}".format)

if OAS_PATH is None or JPG_PATH is None:
    # Demo mode: write synthetic files. The generators live in the repo's tests/ folder.
    sys.path.insert(0, str(Path.cwd().parent / "tests"))
    from sample_design import write_contact_array
    from sample_sem import write_sem_like

    demo_dir = Path(tempfile.mkdtemp())
    OAS_PATH, JPG_PATH = demo_dir / "demo.oas", demo_dir / "demo.jpg"
    # Image: 64 px pitch, 3 nm/px -> 192 nm pitch, matching the design file.
    write_contact_array(OAS_PATH, nx=16, ny=11, pitch_x_nm=192, pitch_y_nm=192, size_nm=84)
    write_sem_like(JPG_PATH, pitch_px=64, radius_px=14, databar_px=64)
    DATABAR_ROWS, PIXEL_SIZE_NM = 64, 3.0
    print("Demo mode: using synthetic files")

Demo mode: using synthetic files


## Design contacts (.oas)

In [3]:
layout = load_layout(OAS_PATH)
print(f"File:          {Path(OAS_PATH).name}")
print(f"Database unit: {layout.dbu * 1000:g} nm")
print(f"Layers:        {list_layers(layout)}")

centers_nm, sizes_nm = read_contacts(layout, *LAYER, cell_name=CELL)
design = pd.DataFrame({
    "x_nm": centers_nm[:, 0], "y_nm": centers_nm[:, 1],
    "w_nm": sizes_nm[:, 0], "h_nm": sizes_nm[:, 1],
})
# Top-to-bottom, left-to-right: the same reading order as the SEM image.
design = design.sort_values(["y_nm", "x_nm"], ascending=[False, True], ignore_index=True)

print(f"\n{len(design)} contacts on layer {LAYER[0]}/{LAYER[1]}")
print(f"x range: {design.x_nm.min():.1f} .. {design.x_nm.max():.1f} nm")
print(f"y range: {design.y_nm.min():.1f} .. {design.y_nm.max():.1f} nm")
design

File:          demo.oas
Database unit: 1 nm
Layers:        [(1, 0), (2, 0)]

176 contacts on layer 1/0
x range: 0.0 .. 2880.0 nm
y range: 0.0 .. 1920.0 nm


,x_nm,y_nm,w_nm,h_nm
0,0.00,1920.00,84.00,84.00
1,192.00,1920.00,84.00,84.00
2,384.00,1920.00,84.00,84.00
3,576.00,1920.00,84.00,84.00
4,768.00,1920.00,84.00,84.00
...,...,...,...,...
171,2112.00,0.00,84.00,84.00
172,2304.00,0.00,84.00,84.00
173,2496.00,0.00,84.00,84.00
174,2688.00,0.00,84.00,84.00


## SEM contacts (.jpg, Otsu threshold)

In [4]:
raw = load_sem_image(JPG_PATH)
image = crop_databar(raw, DATABAR_ROWS)
print(f"File:  {Path(JPG_PATH).name}")
print(f"Image: {raw.shape[1]} x {raw.shape[0]} px; {DATABAR_ROWS} data-bar rows ignored "
      f"-> {image.shape[1]} x {image.shape[0]} px used")

found = detect_contacts(image, dark_contacts=DARK_CONTACTS, min_area_px=MIN_AREA_PX)
sem = pd.DataFrame({
    "u_px": found.centers[:, 0], "v_px": found.centers[:, 1], "area_px": found.areas,
})
if PIXEL_SIZE_NM:
    # Assumes the tile centre is the centre of the image area left after removing the data bar.
    xy = pixel_to_tile_nm(found.centers, image.shape, PIXEL_SIZE_NM)
    sem["x_tile_nm"], sem["y_tile_nm"] = xy[:, 0], xy[:, 1]

print(f"\nOtsu threshold: {found.threshold:.0f} (grey level)")
print(f"{len(sem)} contacts found (listed in image scan order, top to bottom)")
if len(sem):
    median = sem.area_px.median()
    print(f"Area: median {median:.0f} px, range {sem.area_px.min():.0f} .. {sem.area_px.max():.0f} px")
    odd = sem[(sem.area_px < 0.5 * median) | (sem.area_px > 1.5 * median)]
    if len(odd):
        print(f"Check these {len(odd)} contacts, their area is far from the median:")
        print(odd.to_string())
sem

File:  demo.jpg
Image: 1024 x 768 px; 64 data-bar rows ignored -> 1024 x 704 px used

Otsu threshold: 82 (grey level)
176 contacts found (listed in image scan order, top to bottom)
Area: median 497 px, range 491 .. 500 px


,u_px,v_px,area_px,x_tile_nm,y_tile_nm
0,32.00,32.00,497.00,-1438.50,958.50
1,95.97,32.03,495.00,-1246.60,958.40
2,159.98,32.05,497.00,-1054.56,958.36
3,224.04,32.01,496.00,-862.37,958.46
4,288.01,31.98,496.00,-670.46,958.57
...,...,...,...,...,...
171,735.98,671.99,496.00,673.43,-961.46
172,800.03,672.03,495.00,865.60,-961.60
173,864.02,671.93,495.00,1057.56,-961.30
174,927.98,672.05,497.00,1249.44,-961.64


## Summary

In [5]:
print(f"Design contacts: {len(design)}")
print(f"SEM contacts:    {len(sem)}")
if PIXEL_SIZE_NM and len(sem):
    print(f"SEM extent (tile-local): x {sem.x_tile_nm.min():.1f} .. {sem.x_tile_nm.max():.1f} nm, "
          f"y {sem.y_tile_nm.min():.1f} .. {sem.y_tile_nm.max():.1f} nm")
print(f"Design extent:           x {design.x_nm.min():.1f} .. {design.x_nm.max():.1f} nm, "
      f"y {design.y_nm.min():.1f} .. {design.y_nm.max():.1f} nm")

# To save the tables, uncomment:
# design.to_csv("design_contacts.csv", index=False)
# sem.to_csv("sem_contacts.csv", index=False)

Design contacts: 176
SEM contacts:    176
SEM extent (tile-local): x -1438.7 .. 1441.5 nm, y -961.6 .. 958.6 nm
Design extent:           x 0.0 .. 2880.0 nm, y 0.0 .. 1920.0 nm


## Optional: check the contours visually

Opens the interactive SEM viewer with the detected contours (green) and centres (red `+`).
It needs a display, and the notebook waits until you close the window.

In [6]:
SHOW_VIEWER = False

if SHOW_VIEWER:
    import pyqtgraph as pg
    from affine_ransac.view_sem import build_window

    app = pg.mkQApp()
    win = build_window(JPG_PATH, found)
    win.show()
    app.exec()